# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup

**Method, step by step:**

1. **Finds the data.** In Google Colab, the notebook starts on an empty machine, so it downloads a copy of the project (`git clone --depth 1` fetches only the latest version, which is faster). Run locally, it starts inside `work/notebooks/`, so it steps up one folder at a time until it reaches the project's top folder, where `data/raw/` lives.
2. **Loads the tools.** `numpy` does the maths, `pandas` handles tables, and `scipy`'s `poisson` is the probability model used later to test whether a page's missing clicks could be bad luck.
3. **Loads the dataset** into a table called `df`, stopping with a clear message if the file is missing.
4. **Defines `show()` and `show_table()`**, two small helpers that display every result as a titled table in the same layout, so each cell's output is easy to scan.


In [ ]:
import os
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import poisson

# Colab starts empty, so clone the project there; locally, walk up to the folder that holds data/raw.
if "google.colab" in sys.modules:
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/neilivanorencia/flyrank-ml-internship"], check=True)
    os.chdir("flyrank-ml-internship")
else:
    while not os.path.isdir("data/raw") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")

DATA = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(DATA), "Starter CSV not found: run this notebook from inside the project folder."
df = pd.read_csv(DATA)
pd.set_option("display.width", 160)
TIERS = ["top_3", "page_1", "striking", "page_3_5", "deep"]


NUMBER = r"^[-+]?[\d,]*\.?\d+(?:%|x)?$"


def show_table(title: str, frame: pd.DataFrame) -> None:
    """Display a DataFrame as a titled Markdown table.

    Markdown tables look the same in Jupyter, VS Code, Colab, and GitHub, and they avoid the extra
    index row and interactive-table scripts that a plain DataFrame display adds. Columns whose
    cells are all numbers are right-aligned.
    """
    body = frame.reset_index() if frame.index.name or isinstance(frame.index, pd.MultiIndex) else frame
    cells = body.astype(str).apply(lambda col: col.str.replace("|", "\\|", regex=False).str.replace("\n", " ", regex=False))
    align = ["---:" if cells[name].str.strip().str.match(NUMBER).all() else "---" for name in cells.columns]
    lines = ["| " + " | ".join(map(str, cells.columns)) + " |", "| " + " | ".join(align) + " |"]
    lines += ["| " + " | ".join(row) + " |" for row in cells.itertuples(index=False)]
    display(Markdown(f"**{title}**\n\n" + "\n".join(lines)))


def show(title: str, rows: dict[str, object]) -> None:
    """Display a titled two-column table of measures and their values."""
    show_table(title, pd.DataFrame({"Measure": list(rows), "Value": [str(value) for value in rows.values()]}))


show("Dataset loaded", {
    "pages (rows)": f"{len(df):,}",
    "columns": df.shape[1],
    "clients": df["client_id"].nunique(),
})

## 1. My lane (or freestyle) and why

**Lane 4: CTR / Engagement Opportunity Scoring.** This is my provisional lane.

**Research question:** _Which visible pages are losing the most clicks their search position should earn them, by more than bad luck can explain, and does that loss persist into the following month?_

Why this lane:

- **It is decision-shaped.** The output is a review queue for an editor with limited time, which is the exact problem this track is about.
- **The data measures it directly.** Impressions, clicks, and position are Search Console measurements. Nothing depends on FlyRank's own product flags.
- **The method is mine to design.** The starter pipeline already solves decline scoring (Lane 2). Nothing in `scripts/` estimates expected clicks, separates luck from real gaps, or scores a shortfall, so every choice here has to be argued from the data.

**Where this framing departs from the obvious version.** The obvious Lane 4 recipe compares each page's CTR with the median CTR of its position tier and flags the biggest gaps. Section 3 shows three problems with that on this data: the median gives the wrong expectation, most flagged gaps are bad luck, and a CTR gap ignores how many clicks are actually at stake. This project ranks pages by **missed clicks from real shortfalls** instead: the clicks a page fell short by, counted only when the gap is too large to be luck.


In [ ]:
# Pages with position 0 have no position data, so they are excluded from position-based work.
has_pos = df["avg_position"] > 0
impressions = df.loc[has_pos, "impressions_90d"].sort_values(ascending=False)
top10_share = impressions.head(int(len(impressions) * 0.10)).sum() / impressions.sum()

show("How the lane's pages are spread", {
    "pages with real position data": f"{has_pos.sum():,} of {len(df):,}",
    "impressions held by the top 10% of pages": f"{top10_share:.1%}",
    "median impressions per page (90 days)": f"{impressions.median():,.0f}",
    "mean impressions per page (90 days)": f"{impressions.mean():,.0f}",
})

## 2. The question: decision, action, cost of a wrong call

**Decision improved:** which 50 pages a content editor opens this review cycle.

**Who acts:** a content or SEO editor who can work through about 50 pages per cycle. That number sets the main measure of success for the whole project: Precision@50, the share of the 50 recommended pages that really needed attention. Precision@20 is reported too, for a smaller review cycle.

**The action, by pattern:**

| What the data shows                                                     | What the editor does                                                               |
| ----------------------------------------------------------------------- | ---------------------------------------------------------------------------------- |
| A real click shortfall, and visitors who do arrive stay engaged         | Rewrite the title, meta description, or snippet: the page is not winning the click |
| A real click shortfall and weak engagement, where analytics data exists | Check whether the page matches what searchers want before touching the snippet     |
| A shortfall that luck explains                                          | Leave the page alone and keep watching it                                          |

**What a wrong call costs:**

- **A false alarm caused by luck** wastes editor time and risks rewriting a title that works. It also has a quieter cost: a page whose low CTR was luck usually drifts back up on its own, so the rewrite _looks_ like it worked. Over time that teaches the team the wrong lessons about what helps.
- **A missed page** keeps leaking clicks every day, and the loss grows with its traffic.
- Both mistakes are best counted in **clicks**, not pages, which is why the evaluation also reports how many of the lost clicks the queue captures.

**Why data or ML helps at all:** what a page _should_ earn depends on its position, its client, and its content type, and how far a gap can be trusted depends on the page's traffic. Those interact too much for a single fixed threshold; Week 2 shows the evidence.

**The frame in one paragraph:** For a content editor deciding which 50 pages to review this cycle, I will build a ranked queue from Search Console signals, scoring each page by its missed clicks from real shortfalls and measuring success with Precision@50 against what actually happens in a later month. A wrong call costs editor hours and misleading lessons, or clicks left on the table. A plain CTR threshold is not enough because expected CTR shifts with position, client, and content type, and small pages are dominated by luck. I will claim only observed, directional, decision-support results.


**Method, step by step:**

1. Works out each tier's **pooled CTR** (its total clicks ÷ total impressions).
2. Gives every page its **expected clicks**: its impressions × its tier's pooled CTR.
3. Runs the **luck test**: using the Poisson model, how often would a page clicking at the normal rate get this few clicks by chance? That probability is **p**.
4. Calculates **missed clicks** and counts the pages with a **real shortfall** (p below 0.01).


In [ ]:
# Expected clicks come from each tier's pooled CTR, so large pages carry their real weight.
pos = df[has_pos].copy()
pooled = pos.groupby("position_tier")["clicks_90d"].sum() / pos.groupby("position_tier")["impressions_90d"].sum()
pos["expected_clicks"] = pos["impressions_90d"] * pos["position_tier"].map(pooled)
pos["p_shortfall"] = poisson.cdf(pos["clicks_90d"], pos["expected_clicks"])
pos["missed_clicks"] = (pos["expected_clicks"] - pos["clicks_90d"]).clip(lower=0)
credible = pos[pos["p_shortfall"] < 0.01]

show("Real shortfalls versus editor capacity", {
    "editor capacity per review cycle": "50 pages",
    "pages with a real shortfall (p < 0.01)": f"{len(credible):,}",
    "clicks those pages missed (90 days)": f"{credible['missed_clicks'].sum():,.0f}",
    "clicks all pages earned (90 days)": f"{pos['clicks_90d'].sum():,}",
})

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.